# Raw Data Vector Preparation
Creates three dictionaries mapping Pokemon, items, and moves to their raw data vectors for transformer embedding supplementation.

In [ ]:
import pickle
import numpy as np
import pandas as pd

In [ ]:
with open('pokemon_dict.pkl', 'rb') as f:
    pokemon_dict = pickle.load(f)
with open('moves_dict.pkl', 'rb') as f:
    moves_dict = pickle.load(f)
with open('weaknesses.pkl', 'rb') as f:
    weaknesses = pickle.load(f)
with open('resistances.pkl', 'rb') as f:
    resistances = pickle.load(f)
with open('immunities.pkl', 'rb') as f:
    immunities = pickle.load(f)

items_df = pd.read_csv('items_dict_classified.csv')
moves_classified_df = pd.read_csv('moves_classified_full.csv')

In [ ]:
TYPE_ORDER = [
    'Normal', 'Fighting', 'Flying', 'Poison', 'Ground', 'Rock',
    'Bug', 'Ghost', 'Steel', 'Fire', 'Water', 'Grass',
    'Electric', 'Psychic', 'Ice', 'Dragon', 'Dark', 'Fairy'
]
TYPE_TO_IDX = {t: i for i, t in enumerate(TYPE_ORDER)}

## Pokemon Vectors (42 dimensions)
18-dim type one-hot | 18-dim typechart defensive encoding | 6-dim normalized base stats

In [ ]:
pokemon_names = list(pokemon_dict.keys())

type_onehots = []
typechart_encodings = []
base_stats_raw = []

for name in pokemon_names:
    poke = pokemon_dict[name]
    poke_types = poke['type'] if isinstance(poke['type'], list) else [poke['type']]

    # 18-dim type one-hot
    onehot = [0] * 18
    for t in poke_types:
        onehot[TYPE_TO_IDX[t]] = 1
    type_onehots.append(onehot)

    # 18-dim typechart defensive encoding
    matchup = [1.0] * 18
    for t in poke_types:
        if t in weaknesses:
            for weak_type in weaknesses[t]:
                matchup[TYPE_TO_IDX[weak_type]] *= 2.0
        if t in resistances:
            for resist_type in resistances[t]:
                matchup[TYPE_TO_IDX[resist_type]] *= 0.5
        if t in immunities:
            for immune_type in immunities[t]:
                matchup[TYPE_TO_IDX[immune_type]] = 0.0
    typechart_encodings.append(matchup)

    # 6-dim base stats
    stats = [
        poke['hp'], poke['attack'], poke['defense'],
        poke['sp_attack'], poke['sp_defense'], poke['speed']
    ]
    base_stats_raw.append(stats)

type_onehots = np.array(type_onehots)
typechart_encodings = np.array(typechart_encodings)
base_stats_raw = np.array(base_stats_raw, dtype=float)

In [ ]:
# Normalize typechart: divide by 2
typechart_normalized = typechart_encodings / 2.0

# Normalize base stats: column-wise z-scores
stats_mean = base_stats_raw.mean(axis=0)
stats_std = base_stats_raw.std(axis=0)
base_stats_normalized = (base_stats_raw - stats_mean) / stats_std

# Concatenate: type_onehot (18) + typechart (18) + stats (6) = 42
pokemon_vectors = np.concatenate(
    [type_onehots, typechart_normalized, base_stats_normalized], axis=1
)

pokemon_vector_dict = {name: pokemon_vectors[i].tolist() for i, name in enumerate(pokemon_names)}

print(f'Pokemon vector count: {len(pokemon_vector_dict)}')
print(f'Vector dimensionality: {len(next(iter(pokemon_vector_dict.values())))}')
example = list(pokemon_vector_dict.keys())[0]
print(f'Example ({example}): {pokemon_vector_dict[example]}')

## Item Vectors (6 dimensions)
Item category one-hot from items_dict_classified.csv

In [ ]:
category_cols = [c for c in items_df.columns if c not in ['Item Name', 'Description']]

item_vector_dict = {}
for _, row in items_df.iterrows():
    item_vector_dict[row['Item Name']] = [int(row[c]) for c in category_cols]

print(f'Item vector count: {len(item_vector_dict)}')
print(f'Vector dimensionality: {len(next(iter(item_vector_dict.values())))}')
print(f'Category columns: {category_cols}')
example = list(item_vector_dict.keys())[0]
print(f'Example ({example}): {item_vector_dict[example]}')

## Move Vectors (44 dimensions)
18-dim type one-hot | 3-dim category one-hot | 1-dim normalized power | 1-dim normalized priority | 5-dim targets one-hot | 16-dim move category one-hot

In [ ]:
CATEGORY_ORDER = ['Physical', 'Special', 'Status']
CAT_TO_IDX = {c: i for i, c in enumerate(CATEGORY_ORDER)}

TARGET_GROUPS = {
    'Selected Target': 0,
    'Random Target': 0,
    'Previous opponent': 0,
    'Self': 1,
    'All Adjacent Foes': 2,
    'All Adjacent Opponents': 2,
    'All opponents': 2,
    'Enemy Side': 2,
    "Opponent's Side": 2,
    'All Adjacent Pokémon': 3,
    'All': 3,
    'Field': 3,
    'Special': 3,
    'Unknown': 3,
    'Ally': 4,
    'Adjacent Ally': 4,
    'Self or Ally': 4,
    'Team': 4,
    'All allies': 4,
    'Allies': 4,
}

moves_classified_df = moves_classified_df.set_index('Move')
move_cat_cols = list(moves_classified_df.columns[moves_classified_df.columns.get_loc('Damage'):])

In [ ]:
move_names = list(moves_dict.keys())

move_type_onehots = []
move_cat_onehots = []
powers_raw = []
priorities_raw = []
target_onehots = []
move_class_vectors = []

for name in move_names:
    move = moves_dict[name]

    # 18-dim type one-hot
    onehot = [0] * 18
    move_type = move['type']
    if move_type in TYPE_TO_IDX:
        onehot[TYPE_TO_IDX[move_type]] = 1
    move_type_onehots.append(onehot)

    # 3-dim category one-hot
    cat_oh = [0] * 3
    cat_oh[CAT_TO_IDX[move['category']]] = 1
    move_cat_onehots.append(cat_oh)

    # Raw power and priority
    power = move['power'] if isinstance(move['power'], (int, float)) else 0
    powers_raw.append(power)
    priorities_raw.append(move['priority'])

    # 5-dim targets one-hot
    tgt_oh = [0] * 5
    tgt_oh[TARGET_GROUPS[move['targets']]] = 1
    target_onehots.append(tgt_oh)

    # 16-dim move classification from CSV
    row = moves_classified_df.loc[name]
    move_class_vectors.append([int(row[c]) for c in move_cat_cols])

move_type_onehots = np.array(move_type_onehots)
move_cat_onehots = np.array(move_cat_onehots)
powers_raw = np.array(powers_raw, dtype=float)
priorities_raw = np.array(priorities_raw, dtype=float)
target_onehots = np.array(target_onehots)
move_class_vectors = np.array(move_class_vectors)

In [ ]:
# Normalize power: z-score
power_mean = powers_raw.mean()
power_std = powers_raw.std()
powers_normalized = ((powers_raw - power_mean) / power_std).reshape(-1, 1)

# Normalize priority: divide by 5
priorities_normalized = (priorities_raw / 5.0).reshape(-1, 1)

# Concatenate: type(18) + cat(3) + power(1) + priority(1) + targets(5) + move_class(16) = 44
move_vectors = np.concatenate([
    move_type_onehots, move_cat_onehots,
    powers_normalized, priorities_normalized,
    target_onehots, move_class_vectors
], axis=1)

move_vector_dict = {name: move_vectors[i].tolist() for i, name in enumerate(move_names)}

print(f'Move vector count: {len(move_vector_dict)}')
print(f'Vector dimensionality: {len(next(iter(move_vector_dict.values())))}')
example = list(move_vector_dict.keys())[0]
print(f'Example ({example}): {move_vector_dict[example]}')

## Save Dictionaries

In [ ]:
with open('pokemon_vectors.pkl', 'wb') as f:
    pickle.dump(pokemon_vector_dict, f)
with open('item_vectors.pkl', 'wb') as f:
    pickle.dump(item_vector_dict, f)
with open('move_vectors.pkl', 'wb') as f:
    pickle.dump(move_vector_dict, f)

print('Saved: pokemon_vectors.pkl, item_vectors.pkl, move_vectors.pkl')